# Can teacher training copy a fine-tune with no trigger?

*E37, `experiments/teacher_spread`, 2026-10-26. Record: `reports/analyses/2026-10-26_analysis_teacher-spread.md`.
Follows E34, `reports/analyses/2026-10-23_analysis_teacher-training.md`, and uses E36's lowercase fine-tune. One run,
provisional.*

> **Something unexpected happened that I cannot explain.** Two filed predictions reversed. We predicted the lowercase
> copy would pass the adversarial check, as E34's copies of trigger fine-tunes had. It read 2.20 times the unchanged
> model, above the fail line of 1.61. We predicted the adjoint split would lose most of the behaviour; it
> kept 0.80 of the gain and passed at 1.05.

## The answer

Not as filed. Teacher training copied the lowercase fine-tune almost exactly: the copy keeps all of its held-out gain
and does the same damage elsewhere. But the standard mask adversary hurts it twice as much as the unchanged model,
past the fail line, at both batch orders and from ordinary text. Leaving the fine-tune in Δ reads
worse still, so judged against do-nothing the copy passes. The adjoint split, which we expected to lose the behaviour,
kept most of it and passed.

## What we were trying to test

**The model.** The released 67M Pile model with its VPD decomposition. Each matrix is $W = UV^\top + \Delta$; the
columns of $U$ and $V$ are the alive subcomponents' write and read vectors. Γ and Δ are frozen throughout.

**The teacher.** E36's lowercase fine-tune. All 24 decomposed matrices were trained on the next-token loss at every
position of lowercased Pile text, where capitals never come. Adam at rate 3e-5, 300 steps. Its update is
$\delta W$. Every fine-tune that teacher training copied before acted at one trigger token; this one changes
predictions everywhere, as post-training of a larger model would.

**Teacher training** trains the alive $U$ and $V$ to minimise $\mathrm{KL}(\text{teacher} \,\|\, \text{student})$ on
unlabelled text, Γ and Δ frozen, for 1,000 steps with E34's recipe.

**Six states**, each with the alternative it rules out:

- the **unchanged model**, the floor every ratio is taken against;
- **do-nothing**: $\delta W$ goes into Δ, the subcomponents stay as released. The reference: any map should do at
  least this well;
- the **adjoint split** of $\delta W$, the cheap closed-form map, which lost the trigger behaviours in E34;
- the **lowercase copy**, trained on the lowercase rows the teacher trained on;
- the **plain-text copy**, the same documents with capitals kept: is ordinary text enough to copy a style?
- the **lowercase copy at a second batch order**: the copy's noise, and whether two copies land on the same weights.

**What is read**, one symbol per line:

- **behaviour kept** $s$: the state's gain in mean log-probability on held-out lowercase text, over the teacher's.
  0 is the unchanged model, 1 the teacher;
- **teacher gap share**: $\mathrm{KL}(\text{teacher} \,\|\, \text{state})$ over the unchanged model's. 0 is the teacher;
- **adversarial ratio**: an adversary searches 20 steps for the masks on the subcomponents that pull the masked
  model furthest from the target. The ratio is its KL over the unchanged model's; 1 is no more fragile;
- **cosine** between two weight changes, over all 24 matrices at once.

The adversarial check passes at or under 1.40 and fails at or over 1.61 on this card; between is inconclusive. The
pending anchor card asks whether a state should instead pass when it is no worse than its do-nothing.

## What we expected, and why

- **The lowercase copy keeps at least 0.9 of the gain** and leaves at most a tenth of the gap. Every E34 copy
  reached its teacher.
- **It passes the adversarial check.** All of E34's Adam copies did, even where their teachers did not.
- **Its damage on plain text is half to twice the teacher's**: a faithful copy shares the side effects.
- **The adjoint split keeps under half the gain.** On five of E34's six trigger teachers it kept a quarter or less.
- **The plain-text copy also keeps the behaviour**, at low confidence.
- **Two batch orders land on the same weight change**, a cosine of at least 0.8; under 0.5 would mean one of many.

## What we observed

### Does the copy keep the behaviour, and is it faithful?

![behaviour and faithfulness](../experiments/teacher_spread/figures/behaviour_faithful.png)

**Reading it.** It keeps the behaviour and fails the adversarial check. Both lowercase copies, in blue, sit at the
right edge with the teacher, keeping 0.998 and 0.994 of the gain. Both sit above the red fail line, at 2.20 and 1.88.
The plain-text copy, orange, keeps 0.92 and also fails, at 1.74. Do-nothing, black, reads 4.87, so every copy is less
than half as fragile as leaving the update in Δ. The adjoint split, grey, keeps 0.80 of the gain and sits at 1.05,
inside the unchanged model's own spread.

The copy is close to the teacher in every other way. It leaves 0.04 of the gap and does the teacher's damage on plain
Pile text. The adjoint split leaves 0.41 of the gap and a fifth of the damage. So the adjoint split raises the
log-probability of lowercase text without matching the teacher's predictions, while the copy matches them.

By the filed fixed line every copy fails. Judged against do-nothing every copy passes: the lowercase copy reads 0.45
of do-nothing's ratio, the plain-text copy 0.36.

One caveat bears on the failure. The copies were still learning: their training KL fell by a third between steps 500
and 1,000. The 2.20 is a reading of an unfinished copy at one step, and this run cannot say which way more training
moves it.

| state | behaviour kept | gap share | adversarial | against do-nothing |
|---|---|---|---|---|
| do-nothing | 1 | 0 | 4.87 | 1 |
| adjoint split | 0.80 | 0.41 | 1.05 | 0.22 |
| lowercase copy | 0.998 | 0.04 | 2.20 | 0.45 |
| lowercase copy, 2nd order | 0.994 | 0.04 | 1.88 | 0.39 |
| plain-text copy | 0.92 | 0.08 | 1.74 | 0.36 |

### Is the copies' failure the harness's alone?

![two adversaries](../experiments/teacher_spread/figures/adversaries.png)

**Reading it.** No: a second adversary also puts both copies above its line. The hollow diamonds are the budget
adversary, the same search on half the rows, read under the masks it finds. It reads the lowercase copy at 1.80 and
the plain-text copy at 1.66. Both are above its line of 1.241. E34's copies of trigger fine-tunes read 0.90 to 1.37 under either
adversary, the grey band. Do-nothing reads 3.92 and the adjoint split 1.17. The two adversaries rank the states alike;
the budget adversary reads the copies and do-nothing about a fifth lower than the harness.

### Do two copies land on the same weight change?

![placement](../experiments/teacher_spread/figures/placement.png)

**Reading it.** Yes. The two batch orders' weight changes have a cosine of 0.93, above the 0.8 line, and no matrix
reads under 0.91. The lowercase and plain-text copies read 0.81 against each other. Each copy points along the
fine-tune at 0.44, level with the adjoint split. E34's copies read 0.08 to 0.29 against their fine-tunes. A random direction reads
zero. Between batch orders the behaviour kept moved by 0.004. The adversarial ratio moved by 14%, inside the spread E34 saw.

## What we learned

Teacher training can copy a broad fine-tune: the copy matches the teacher's predictions to a twentieth of the
original gap and shares its side effects. But the copy is not faithful by the fixed line, under two adversaries, at two
batch orders and from two kinds of text. E34's finding that a copy beats the update left in Δ
still holds, by a factor of two. Its finding that a copy is as faithful as the original decomposition does not carry
over from trigger fine-tunes to this broad one. The adjoint split keeps most of a broad gain and passes, though it
matches the teacher far less closely.

## How compelling is the evidence

- **One teacher, one fine-tune seed, one synthetic style**, read at one step on copies still learning.
- **The failure holds across everything varied**: two batch orders, two kinds of text, two adversaries.
- **One adversary seed per updated state.** A copy's batch-order spread in the ratio is 14% here and up to 25% in
  E34. The lowercase copy sits 37% above the fail line; the plain-text copy only 8% above it.
- **Behaviour kept is a mean over positions.** The adjoint split shows the mean can be kept with different
  predictions.
- **Passing against do-nothing is not a filed rule**; it waits on the anchor decision.

## Surprises

- **The copy fails the adversarial check, against a filed prediction that it would pass.** All of E34's Adam copies
  had read under 1.37. We have no tested cause. The copy was still learning, the update is broad and the teacher is
  larger than E34's; none of these was varied.
- **The adjoint split keeps the behaviour, against a filed prediction that it would lose it.** It keeps 0.80 of the
  gain. Its gap is about ten times the copy's. We have no tested cause.
- **The copies point along the fine-tune more than E34's did**, 0.44 against an expected 0.08 to 0.3.
- **The harness reads the updated states above the budget adversary**, the other way round from E34's trigger
  teachers. E34 Amendment 5 is reading why the two disagree.
- **The copies are three quarters of the teacher's size**, not the expected half.

## What next

The options on card `decision-broad-copy`:

1. **Train the copy three times longer**, about $2, reading the adversarial check at several steps. A ratio falling
   under the fail line as the loss flattens means under-training; one that stays means the failure is the copy's.
2. **Judge the copy against do-nothing**, option b of the anchor card. Every copy here then passes.
3. **Take the adjoint split as the map for broad updates, or start the copy from it.** A copy started there and read
   at several steps would show whether matching the teacher brings the fragility.
4. **Test whether the frozen Γ or the subcomponents' span limits the copy**, E15's refit at about $9. A copy under
   the line with a refit Γ would mean Γ must move with the update.

## Scorecard

| id | claim | read | verdict |
|---|---|---|---|
| P1 | lowercase copy keeps the behaviour, s ≥ 0.9 | 0.998 | holds |
| P2 | lowercase copy close to the teacher, gap share ≤ 0.10 | 0.044 | holds |
| P3 | lowercase copy passes the harness, ≤ 1.40 | 2.20, fail line 1.61; 0.45 of do-nothing | **fails, reversal** |
| P4 | damage 0.5 to 2 times the teacher's | 0.996 | holds |
| P5 | adjoint split loses the behaviour, s < 0.5 | 0.804 | **fails, reversal** |
| P6 | plain-text copy keeps the behaviour, s ≥ 0.9 | 0.919 | holds |
| P7 | one placement, cosine ≥ 0.8 | 0.929 | holds |

All three controls passed. The unchanged model read its known floor, 0.0748. The teacher reproduced E36's gain within
0.0001, and do-nothing read exactly as the teacher.

## Not run

- A second fine-tune seed or a second broad teacher.
- Copies read at other steps, or trained longer.
- The plain-text copy at a second batch order; the second batch order under the budget adversary.
- A second adversary seed on any updated state; adversaries past 20 steps.
- A copy started from the adjoint split; E15's Γ refit.
- LoRA teachers on this task; the larger model.

**Provenance.** RunPod H100 80GB HBM3, about 21 min and $1.40, job 20:22 to 20:36, exit 0. Results
`experiments/teacher_spread/results/teacher_spread.json`; predictions `results/predictions.json`, filed 19:53:47,
unedited; figures from `experiments/teacher_spread/plot.py`. Full tables, block readings and hashes are in the report.